# 24 — Why RF amplitude lengthens drift time in an rf-confining drift cell (SRIG)

**Who this is for.** A 1st/2nd-year graduate student meeting RF ion guides for the first time. Companion to nb23 (TW-SLIM): the two notebooks ask the same experimental question, *why do ions arrive later when I raise the RF?*, on two very different devices.

## Background

A **stacked-ring ion guide** (SRIG) confines ions with an RF voltage applied in alternating phase to a stack of ring electrodes, while a DC ladder superimposed on the rings drives them axially through a buffer gas. The model device follows **Allen & Bush (J. Am. Soc. Mass Spectrom. 27, 2054–2063, 2016)**: rings of 5 mm i.d. at a 1.5 mm pitch, alternating RF at 2.8 MHz, a uniform DC gradient, helium at 3.5 Torr, and the peptide ion GRGDS2+. They measured drift times up to 10% longer than the electrostatic drift equation predicts at the lowest drift field (E/P 0.35 V cm⁻¹ Torr⁻¹), growing with RF amplitude, in the 5 mm cell but hardly at all in a 7 mm cell.

The textbook picture of RF confinement is the **pseudopotential** (Dehmelt, Gerlich): averaging over the fast RF gives an effective potential Φ* ∝ E²_RF/(m ω²) that pushes ions away from the electrodes. A drift cell adds one complication. Because the rings are discrete, E_RF, and therefore Φ*, varies along the axis with the ring pitch, including on the axis itself (the I₀² term in Tolmachev's expression). A drifting ion crosses that ripple once per pitch while the DC ramp pushes it forward.

Allen and Bush attribute the longer drift times to "pseudopotential wells generated by alternating rf potentials" and to ions "spending more time in lower field regions of the drift cell". Their picture of the wells (their Fig. 4b) is drawn with the drift voltage switched off. With the drift field on, the useful question is whether the ripple is ever steep enough to cancel the DC push. Where it is not, the summed potential has no low point and no ion can be held, yet the crossing is uneven: the ion moves slowly through the half pitch where the ripple opposes the ramp and quickly through the half where it adds to it, so it spends more of its time where the net field is weak and its average velocity falls. Where the ripple is steeper than the ramp, real minima exist and ions can be held. The ripple deepens away from the axis, so which of the two applies depends on where the ions travel.

**How this notebook tests it.** Section 1.6 turns the solved fields into a prediction with no adjustable parameter. For an ion held at one radius, overdamped drift plus diffusion over the potential at that radius has an exact mean velocity (the Stratonovich result for a tilted periodic potential), which gives a predicted slowdown at every radius. Two averages over the cross section turn that into one expected delay per (E/P, Vpp) cell: one weights each radius by how often the flown ions crossed a pitch there, the other assumes thermal equilibrium across the section within each pitch (the Fick–Jacobs reduction). Section 2 flies the same cells and section 2b sets the flown delays against the predictions. Path length and kinetic energy are recorded alongside as controls on the two other candidate explanations, longer paths and RF heating.

**Ion and gas.** GRGDS2+ (z = +2) in 3.5 Torr He. The ion mass and hard-sphere cross section in the parameter cell are the values every flight uses, each named there with its status. Every delay in this notebook is a ratio to an RF-off reference cell flown with the same ion and settings, so a bias in the cross section cancels to first order.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
# TEXT SIZE (run me first): floors all rendered markdown text and
# equations at the ### (h3) size for legibility 
# stamped by the development tree's text-size tool. Styling
# only; no physics, no state.
# WHERE IT APPLIES (stated so an inert run is never a silent surprise):
# JupyterLab / Notebook 7 / classic / nbviewer -- takes effect when
# this cell runs. VS CODE: this cell is INERT by construction -- VS
# Code renders markdown cells natively and sandboxes cell output, so
# injected CSS cannot reach them; there the floor comes from the
# repo's .vscode/settings.json (notebook.markup.fontSize 19 /
# notebook.output.fontSize 14), which applies without running anything.
from IPython.display import HTML, display
display(HTML("""<style>
/* JupyterLab / Notebook 7 */
.jp-RenderedMarkdown p, .jp-RenderedMarkdown li,
.jp-RenderedMarkdown td, .jp-RenderedMarkdown th,
.jp-RenderedMarkdown blockquote { font-size: 1.3em; }
.jp-RenderedMarkdown .MathJax, .jp-RenderedMarkdown mjx-container,
.jp-RenderedMarkdown .katex { font-size: 130% !important; }
/* classic notebook / nbviewer */
.text_cell_render p, .text_cell_render li,
.text_cell_render td, .text_cell_render th,
.text_cell_render blockquote { font-size: 1.3em; }
.text_cell_render .MathJax, .text_cell_render mjx-container,
.text_cell_render .katex { font-size: 130% !important; }
</style>"""))


## 0. Parameters — every physics choice, named, with its reason

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
from IPython.display import Image, display

from ion_gym.io.paths import examples_dir, outputs_dir, repo_root
from ion_gym.physics import ensemble_driver as ed
from ion_gym.physics.constants import K_B_EV
from ion_gym.viz.viz_core import DEFAULT_DPI

# the shipped manuscript_examples folder holds this notebook and the deck
# generator; figures and the flown bank go to the outputs folder
STUDY = repo_root() / 'manuscript_examples' / 'drift_cell'
OUT = outputs_dir('manuscript_examples', 'drift_cell', '24_srig_rf_mechanism')
OUT.mkdir(parents=True, exist_ok=True)

# ---- device: the Allen & Bush 2016 rf-confining drift cell -------------
BORE_ID_MM = 5.0        # ring inner diameter: the 5 mm cell of their Fig. 2b/2d
N_RINGS = 44            # model length: 33 measured pitches plus entry and exit rings
SOURCE_RING = 3         # ions born half a pitch past this ring's centre
EXIT_RING = N_RINGS - 3 # x_max bound at this ring's centre, where flights end
PLANE_A_RING = 6        # transit starts at this ring's centre plane (entry fringe excluded)
PLANE_B_RING = 39       # and ends at this ring's centre (exit fringe excluded): 33 pitches = 49.5 mm
RF_FREQ_HZ = 2.8e6      # instrument RF
P_TORR = 3.5            # helium pressure of every Allen & Bush measurement
GAS = 'He'
T_GAS_K = 298.0         # gas temperature

# ---- ion: GRGDS [M+2H]2+ ---------------------------------------------------
# Ion MASS (not m/z): 492.5 Da = the protonated [M+2H]2+ ion, m/z 246.3
# (Allen, Giles, Gilbert, Bush, Analyst 141 (2016) 884, ref 38 of Allen &
# Bush 2016); the neutral peptide is 490.5 Da.
ION_MASS_DA = 492.5
ION_Z = 2
# He hard-sphere cross section: 141 A^2 = 1.41e-18 m^2, measured in this
# same RF-confining cell (ref 38, ESI Tables S3/S4). The DC drift-tube
# value is 145.6 A^2 (Kemper et al. 2009), 3% larger.
# replacing the 1.55e-18 placeholder. It cancels to first order in every
# delay ratio below; it does NOT cancel in absolute drift velocities.
SIGMA_M2 = 1.41e-18

# the deck generator's keyword set, built ONCE: every spec in this
# notebook is made from these values, so the operating point printed on
# a figure is the one the solver received
DEVICE = dict(bore_id_mm=BORE_ID_MM, n_rings=N_RINGS,
              rf_freq_hz=RF_FREQ_HZ, p_torr=P_TORR, T_k=T_GAS_K, gas=GAS,
              ion_mass_da=ION_MASS_DA, ion_charge=ION_Z, sigma_m2=SIGMA_M2,
              source_ring=SOURCE_RING, exit_ring=EXIT_RING)

# ---- the two experimental knobs -----------------------------------------
E_OVER_P = [1.51, 0.507, 0.35]   # V/(cm Torr), Allen & Bush's moderate, low and lowest fields
VPP = [0, 15, 50, 100, 200]      # RF peak-to-peak; 0 = RF off. 50 Vpp added between their 15 and 100
SHOW_E_OVER_P = 0.507            # field of the geometry and mechanism figures (sections 1, 1.5)
REF_E_OVER_P = 1.51              # the RF-off reference cell every delay is measured against
REF_VPP = 0

# ---- numerics ----------------------------------------------------------
# The hard-sphere kernel applies at most one collision per step, so a
# step comparable to the mean collision time (5.5 ns for this ion at rest
# at the ruled sigma in 3.5 Torr He) undercounts collisions and flies the
# ion too mobile. The kernel's advisory asks for dt <= tau_c / 20 =
# 0.27 ns. The August campaign flew 0.5 ns (roughly 5% of collisions
# lost). 0.2 ns satisfies the advisory with margin and still resolves the
# 357 ns RF period with ~1800 steps.
DT_NS = 0.2
SEED = 60601
ROWS_PER_ION = 8000     # recording budget: >= 200 rows per pitch for crossing interpolation
N_WORKERS = ed.default_workers()   # every core but two
from ion_gym.physics.sizing import default_record_budget_gb, resources_line
RECORD_BUDGET_GB = default_record_budget_gb()   # GB the trajectory record may
                        # use: the workbench default (RECORD_REFUSE_FRACTION
                        # of RAM). build_run warns at half, refuses above.
print(resources_line(N_WORKERS, RECORD_BUDGET_GB))
N_EXAMPLE = 3           # example trajectories drawn in section 1

# ---- section 1.6 predictor ------------------------------------------
KT_EV = K_B_EV * T_GAS_K
N_FINE = 4096           # fine points per pitch in the predictor integrals
R_GRID_MAX_MM = 2.25    # predictor radii run to 0.25 mm inside the ring bore
R_PLOT_MAX_MM = 1.25    # radius range drawn in the slowdown-vs-radius figure
GAS_DAMPED_PSEUDO = True   # scale the pseudopotential by (w tau)^2/(1+(w tau)^2), tau = mK/q
FJ_WALL_MIN_KT = 10.0   # Fick-Jacobs average refused unless the radial wall at R_GRID_MAX_MM exceeds this

# ---- section 2.0 ion numbers ---------------------------------------
TARGET_DELAY_SE = 0.002    # 0.2% absolute standard error on every flown delay
TAIL_SIGMA = 5.0           # residence-tail excess resolved at this many standard errors
TAIL_P_IG = 0.0030         # planning value: referee fraction of crossings beyond 4 sd (E/P 0.35, 200 Vpp, campaign)
TAIL_P_FLOWN = 0.0048      # planning value: flown fraction beyond 4 sd, same cell
N_TRANSMISSION = 1000      # RF-off cells below the reference field: survival only
SURVIVAL_RF_OFF_PLAN = 0.2 # planning transit fraction with RF off (Allen & Bush: 21% survive 8.4 cm at E/P 1.51), replaced by the banked value when one exists
SURVIVAL_RF_ON_PLAN = 1.0  # planning transit fraction with RF on, replaced by the banked value when one exists

RUN = False   # True flies every cell not yet banked (section 2.0 sizes it, the quote cell prices it)
# Raw campaign records live with the study (they are inputs to the
# figures, not deliverables). The file name carries the step size, and
# the loader refuses cells whose settings differ from this cell's.
DATA = outputs_dir('manuscript_examples', 'drift_cell', 'data')
DATA.mkdir(parents=True, exist_ok=True)
CAMPAIGN_FILE = DATA / f"g1m_dt{DT_NS:g}ns.json"
print(f"campaign file: {CAMPAIGN_FILE.name} | {N_WORKERS} workers | "
      f"dt {DT_NS} ns | sigma {SIGMA_M2:.3g} m^2 | ion {ION_MASS_DA} Da z=+{ION_Z}")

In [ ]:
# ==== the campaign runner (inlined from g1_run.py) ===============
# Rewritten for the manuscript campaign: every flight takes
# the DEVICE values from the parameter cell (the old runner silently
# used the generator's defaults for mass, cross section and gas), ions
# fly on worker threads with progress, and every cell records its pitch
# crossings, so one campaign feeds the transit, residence and radial
# figures alike.
'''G1 -- does RF amplitude lengthen drift time, and what moves with it?
Allen & Bush 2016 rf-confining drift cell, E/P x Vpp grid, per-ion
transit between two interior planes.

Per ion (from the recorded trajectory, plane crossings interpolated):
  transit_us   time from plane a to plane b
  path_mm      arc length flown between the two crossings (path_mm channel)
  e_ax_mean    time-mean of the axial field over the segment (exact kernel integral)
  ke_mean_ev   time-mean kinetic energy over the segment
  r_mean       time-mean radius over the segment
  tau_us       residence time in every ring pitch between the planes
  r_period     mean radius during each of those pitches
Per cell: survivors, mean/SE transit, ATD skew, path per axial mm,
<E_ax>/E_ideal, mean r, and the per-ion lists above.
'''
import math
import sys

# the deck generator lives beside this notebook (make_rf_drift_cell.py);
# this study sweeps geometry, so it calls the generator
sys.path.insert(0, str(STUDY))
from make_rf_drift_cell import rf_drift_cell_spec, PITCH_MM   # noqa: E402
from ion_gym.physics.sim_build import build_run

# the origin guard pins ion_gym but not this sidecar module, so a stale
# generator would otherwise surface cells later as a confusing TypeError;
# refuse here instead, naming the copy that was imported and the remedy
import inspect
import make_rf_drift_cell as generator_module
if "source_ring" not in inspect.signature(rf_drift_cell_spec).parameters:
    raise RuntimeError(
        f"stale deck generator: {generator_module.__file__} lacks the "
        f"source_ring parameter this notebook requires ("
        f"revision). Put the matching make_rf_drift_cell.py beside this "
        f"notebook and RESTART THE KERNEL -- an already-imported "
        f"copy stays cached for the kernel's lifetime.")


def plane_crossing(t, x, x_plane, *cums):
    '''First upward crossing of x_plane: interpolated t and each cumulative
    channel in `cums` at the crossing, plus the row index. None if the ion
    never crosses.'''
    idx = np.nonzero((x[:-1] < x_plane) & (x[1:] >= x_plane))[0]
    if idx.size == 0:
        return None
    i = idx[0]
    f = (x_plane - x[i]) / (x[i + 1] - x[i])
    vals = [c[i] + f * (c[i + 1] - c[i]) for c in cums]
    return dict(t=t[i] + f * (t[i + 1] - t[i]), cums=vals, i=i)


def per_ion(traj, cols, x_a, x_b, period_mm):
    '''One ion's record between the planes, or None if it never reaches
    plane b (lost to a ring or timed out: counted by fate, not here).'''
    c = {n: k for k, n in enumerate(cols)}
    t = traj[:, c["t"]]
    x = traj[:, c["x"]]
    path = traj[:, c["path_mm"]]
    eat = traj[:, c["e_axial_tint"]]
    ket = traj[:, c["ke_tint"]]
    r = traj[:, c["radius"]]
    ncol = traj[:, c["n_col"]]
    a = plane_crossing(t, x, x_a, path, eat, ket, ncol)
    b = plane_crossing(t, x, x_b, path, eat, ket, ncol)
    if a is None or b is None:
        return None
    seg = slice(a["i"] + 1, b["i"] + 1)
    dtr = b["t"] - a["t"]
    d = [bb - aa for aa, bb in zip(a["cums"], b["cums"])]
    out = dict(transit_us=dtr, path_mm=d[0], ncol=d[3],
               e_ax_mean=d[1] / dtr,          # V/mm: tracer_rz integrates V/mm us
               ke_mean_ev=d[2] / dtr,         # eV, exact time mean
               r_mean=float(np.mean(r[seg])),
               r_max=float(np.max(r[seg])))
    # residence time per ring pitch: crossing times of every plane
    # x_a + k*period up to x_b; tau_k = t_{k+1} - t_k, plus the mean
    # radius while in pitch k (micromotion averaged over the pitch)
    planes = np.arange(x_a, x_b + 1e-9, period_mm)
    tc = []
    ic = []
    for xp in planes:
        cr = plane_crossing(t, x, xp)
        if cr is None:
            break
        tc.append(cr["t"])
        ic.append(cr["i"])
    out["tau_us"] = np.diff(np.asarray(tc)).tolist()
    out["r_period"] = [float(np.mean(r[ic[k] + 1: ic[k + 1] + 1]))
                       if ic[k + 1] > ic[k] else float(r[ic[k]])
                       for k in range(len(ic) - 1)]
    return out


def fly_ensemble(fly, n_ions, *, reduce, n_workers, label, every_s=15.0):
    '''Fly n_ions on worker threads. Each worker reduces its trajectory
    to the per-ion record at once, so no trajectory is held in memory.
    Prints done/total, elapsed, rate and ETA. A worker error is re-raised
    with its traceback, and a run that ends short is refused.'''

    def fly_fn(i):
        traj, summ = fly(i)
        rec = dict(summ)
        rec["per_ion"] = reduce(traj)
        return None, rec

    handle = ed.run_threaded(n_ions, fly_fn,
                             check_every=max(2, min(50, n_ions // 20)),
                             keep_traj=False, n_workers=n_workers)
    last_print = time.perf_counter()
    while not handle.done:
        time.sleep(0.2)
        prog = handle.latest
        now = time.perf_counter()
        if prog is not None and prog.done and now - last_print >= every_s:
            print(f"    [{label}] {prog.done}/{prog.total} ions, "
                  f"{prog.elapsed_s:.0f} s, {prog.rate_hz:.2f} ion/s, "
                  f"ETA {prog.eta_s:.0f} s, {prog.n_workers} workers",
                  flush=True)
            last_print = now
    if handle.error is not None:
        raise RuntimeError(
            f"[{label}] a worker failed:\n{handle.error_tb}") from handle.error
    final = handle.final
    if final is None or final.stopped_early or len(final.results) != n_ions:
        got = 0 if final is None else len(final.results)
        reason = "" if final is None else final.stop_reason
        raise RuntimeError(f"[{label}] ensemble ended with {got}/{n_ions} "
                           f"ions ({reason or 'no stop reason recorded'})")
    return sorted(final.results, key=lambda res: res.index)


def run_cell(*, device, e_over_p, vpp, n_ions, dt_ns, plane_a_ring,
             plane_b_ring, seed, rows_per_ion, n_workers):
    '''Fly one (E/P, Vpp) cell and reduce it to the JSON record.'''
    spec = rf_drift_cell_spec(**device, rf_vpp=vpp, e_over_p=e_over_p,
                              n_ions=n_ions, dt_ns=dt_ns, seed=seed)
    # recording stride sized so every ion keeps ~rows_per_ion rows
    spec.integration.rec_every = max(
        int(spec.integration.t_max_us / (dt_ns * 1e-3) / rows_per_ion), 1)
    # measurement planes at ring CENTRES (shape x_mm is the ring's
    # upstream face, so the centre is half a thickness downstream)
    half_ring = 0.5 * spec.geometry.electrodes[0].shapes[0].params["width_mm"]
    x_a = (spec.geometry.electrodes[plane_a_ring - 1].shapes[0].params["x_mm"]
           + half_ring)
    x_b = (spec.geometry.electrodes[plane_b_ring - 1].shapes[0].params["x_mm"]
           + half_ring)
    E_ideal_v_mm = e_over_p * spec.collisions.P_torr / 10.0
    if not (device["source_ring"] < plane_a_ring < plane_b_ring
            < device["exit_ring"]):
        raise ValueError(
            f"measurement planes {plane_a_ring}->{plane_b_ring} must lie "
            f"strictly between the source ring {device['source_ring']} and "
            f"the exit ring {device['exit_ring']}: a plane outside that "
            f"span records nothing")
    t0 = time.time()
    model, fly, cols, births = build_run(spec, record_budget_gb=RECORD_BUDGET_GB)

    def reduce(traj):
        return per_ion(traj, cols, x_a, x_b, PITCH_MM)

    results = fly_ensemble(fly, n_ions, reduce=reduce, n_workers=n_workers,
                           label=f"E/P {e_over_p:g}, {vpp:g} Vpp")
    rows = []
    fates = {}
    for res in results:
        kind = int(res.summary["kind"])
        fates[kind] = fates.get(kind, 0) + 1
        if res.summary["per_ion"] is not None:
            rows.append(res.summary["per_ion"])
    dt_wall = time.time() - t0
    n = len(rows)
    out = dict(bore_id_mm=device["bore_id_mm"], n_rings=device["n_rings"],
               rf_freq_hz=device["rf_freq_hz"], p_torr=device["p_torr"],
               T_k=device["T_k"], gas=device["gas"],
               ion_mass_da=device["ion_mass_da"],
               ion_charge=device["ion_charge"], sigma_m2=device["sigma_m2"],
               source_ring=device["source_ring"],
               exit_ring=device["exit_ring"],
               e_over_p=e_over_p, vpp=vpp, n_ions=n_ions, dt_ns=dt_ns,
               seed=seed, plane_a_ring=plane_a_ring,
               plane_b_ring=plane_b_ring, x_a_mm=x_a, x_b_mm=x_b,
               L_mm=x_b - x_a, E_ideal_v_mm=E_ideal_v_mm, fates=fates,
               n_transit=n, wall_s=dt_wall, n_workers=n_workers,
               notes=spec.notes)
    if n:
        tr = np.array([row["transit_us"] for row in rows])
        pl = np.array([row["path_mm"] for row in rows])
        ea = np.array([row["e_ax_mean"] for row in rows])
        rm = np.array([row["r_mean"] for row in rows])
        nc = np.array([row["ncol"] for row in rows])
        ke = np.array([row["ke_mean_ev"] for row in rows])
        sk = float(((tr - tr.mean()) ** 3).mean() / tr.std() ** 3) if n > 3 \
            else float("nan")
        out.update(dict(
            transit_mean_us=float(tr.mean()),
            transit_se_us=float(tr.std(ddof=1) / math.sqrt(n)) if n > 1 else float("nan"),
            transit_sd_us=float(tr.std(ddof=1)) if n > 1 else float("nan"),
            transit_skew=sk,
            transit_p05_us=float(np.percentile(tr, 5)),
            transit_p50_us=float(np.percentile(tr, 50)),
            transit_p95_us=float(np.percentile(tr, 95)),
            v_d_m_s=float((x_b - x_a) / tr.mean() * 1e3),
            path_per_axial_mm=float(pl.mean() / (x_b - x_a)),
            path_per_us=float((pl / tr).mean()),
            e_ax_over_ideal=float(ea.mean() / E_ideal_v_mm),
            r_mean_mm=float(rm.mean()),
            ke_mean_ev=float(ke.mean()),
            # lab-frame mean KE as an equivalent temperature (3/2 kT):
            # includes drift + micromotion + thermal; a T_eff PROXY only
            T_ke_k=float(ke.mean() / (1.5 * K_B_EV)),
            ncol_per_us=float((nc / tr).mean()),
            transit_us_all=tr.tolist(), path_mm_all=pl.tolist(),
            tau_us_per_ion=[row["tau_us"] for row in rows],
            r_period_per_ion=[row["r_period"] for row in rows]))
    return out


# the settings a banked cell must share with the current parameters
# before it may be reused (anything else would mix two operating points)
CELL_SETTINGS = ("bore_id_mm", "n_rings", "rf_freq_hz", "p_torr", "T_k",
                 "gas", "ion_mass_da", "ion_charge", "sigma_m2",
                 "source_ring", "exit_ring", "dt_ns", "seed",
                 "plane_a_ring", "plane_b_ring", "n_ions")


def cell_mismatch(cell, wanted):
    '''Names every setting where a banked cell differs from `wanted`
    (empty string when they agree).'''
    diffs = []
    for k in CELL_SETTINGS:
        if k not in cell:
            diffs.append(f"{k} not recorded")
        elif cell[k] != wanted[k]:
            diffs.append(f"{k} {cell[k]!r} (now {wanted[k]!r})")
    return ", ".join(diffs)


def run_campaign(out_path, *, device, n_ions_by_cell, dt_ns, e_over_p, vpp,
                 plane_a_ring, plane_b_ring, seed, rows_per_ion, n_workers):
    '''Fly every (E/P, Vpp) cell not already banked in out_path. The file
    is rewritten after each cell, so an interrupted campaign resumes
    where it stopped. A banked cell flown with different settings is
    refused, never silently reused or overwritten.'''
    out_path = Path(out_path)
    cells = json.loads(out_path.read_text()) if out_path.exists() else []
    for ep in e_over_p:
        for v in vpp:
            if (ep, v) not in n_ions_by_cell:
                raise KeyError(f"no ion count for E/P {ep:g}, {v:g} Vpp: "
                               f"run the section 2.0 cell first")
            n = n_ions_by_cell[(ep, v)]
            wanted = dict(device, dt_ns=dt_ns, seed=seed,
                          plane_a_ring=plane_a_ring,
                          plane_b_ring=plane_b_ring, n_ions=n)
            have = [c for c in cells if c["e_over_p"] == ep and c["vpp"] == v]
            if have:
                diff = cell_mismatch(have[0], wanted)
                if diff:
                    raise RuntimeError(
                        f"{out_path.name} already holds E/P {ep:g}, {v:g} Vpp "
                        f"flown with different settings: {diff}. Move the "
                        f"file aside to start a new campaign, or restore "
                        f"the parameters it was flown with.")
                print(f"E/P {ep:g}, {v:g} Vpp: already banked "
                      f"({have[0]['n_ions']} ions), skipping", flush=True)
                continue
            c = run_cell(device=device, e_over_p=ep, vpp=v, n_ions=n,
                         dt_ns=dt_ns, plane_a_ring=plane_a_ring,
                         plane_b_ring=plane_b_ring, seed=seed,
                         rows_per_ion=rows_per_ion, n_workers=n_workers)
            cells.append(c)
            out_path.write_text(json.dumps(cells))
            summary = (f"transit {c['transit_mean_us']:.1f}+-{c['transit_se_us']:.1f} us, "
                       f"<Eax>/E {c['e_ax_over_ideal']:.4f}, "
                       f"path/mm {c['path_per_axial_mm']:.2f}, "
                       f"<r> {c['r_mean_mm']:.2f} mm" if c["n_transit"]
                       else "no ion reached the far plane")
            print(f"E/P {ep:g}, {v:g} Vpp: {c['n_transit']}/{n} transits, "
                  f"fates {c['fates']}, {summary} [{c['wall_s']:.0f} s]",
                  flush=True)
    return cells

In [ ]:
# ==== INLINED from g1_figs.py: figure builders, verbatim
# minus __main__. All rendering stays on ion_gym.viz.design_maps.
"""G1 figures from g1_run JSON cells -- via design_maps.render_profiles
(the sanctioned 1-D design-profile renderer; no inline matplotlib).

Three profiles, one PNG each (caller saves):
  transit ratio   t / t_ideal(E/P) vs Vpp, one curve per E/P
  path per mm     integrated arc length per axial mm vs Vpp, per E/P
  ATD shape       arrival-time histogram (density vs t/median) per Vpp,
                  at the E/P the caller names
t_ideal(E/P) = t(E/P_ref, Vpp=0) * (E/P_ref)/(E/P): the same model, same
dt, same ion, so the dt timing bias and any cross-section bias cancel in
the ratio (common mode); the reference cell must be RF-off.
"""
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np

from ion_gym.viz.design_maps import ProfileCurve, render_profiles


def load_cells(*paths):
    cells = []
    for p in paths:
        cells += json.loads(Path(p).read_text())
    return [c for c in cells if c.get("n_transit", 0) > 0]


def op_transit(c):
    return (f"Allen&Bush cell i.d. {c['bore_id_mm']:g} mm, {c['n_rings']}-ring "
            f"model, planes {c['x_a_mm']:.2f}->{c['x_b_mm']:.2f} mm "
            f"(L {c['L_mm']:.1f} mm), {c['rf_freq_hz']/1e6:g} MHz, "
            f"{c['p_torr']:g} Torr {c['gas']} {c['T_k']:g} K, GRGDS2+ "
            f"{c['ion_mass_da']:g} Da z=+{c['ion_charge']}, HS1 sigma "
            f"{c['sigma_m2']:.3g} m^2 (cancels in ratios), dt {c['dt_ns']:g} ns, "
            f"n <= {c['n_ions']}/cell")


def transit_ratio_curves(cells, *, ref_ep, ref_vpp=0.0):
    ref = [c for c in cells if abs(c["e_over_p"] - ref_ep) < 1e-9
           and c["vpp"] == ref_vpp]
    if len(ref) != 1:
        raise ValueError(f"need exactly one RF-off reference cell at E/P "
                         f"{ref_ep}, found {len(ref)}")
    t_ref = ref[0]["transit_mean_us"]
    curves, se = [], []
    for ep in sorted({c["e_over_p"] for c in cells}, reverse=True):
        cs = sorted([c for c in cells if c["e_over_p"] == ep and c["n_transit"] >= 5],
                    key=lambda c: c["vpp"])
        t_ideal = t_ref * ref_ep / ep
        x = [c["vpp"] for c in cs]
        y = [c["transit_mean_us"] / t_ideal for c in cs]
        e = [c["transit_se_us"] / t_ideal for c in cs]
        curves.append(ProfileCurve(f"E/P {ep:g} V/cm/Torr", x, y,
                                   {"marker": "o"}, yerr=e))
        se.append((x, y, e))
    return curves, se


def path_curves(cells):
    curves = []
    for ep in sorted({c["e_over_p"] for c in cells}, reverse=True):
        cs = sorted([c for c in cells if c["e_over_p"] == ep and c["n_transit"] >= 5],
                    key=lambda c: c["vpp"])
        curves.append(ProfileCurve(f"E/P {ep:g} V/cm/Torr",
                                   [c["vpp"] for c in cs],
                                   [c["path_per_axial_mm"] for c in cs],
                                   {"marker": "s"}))
    return curves


def atd_curves(cells, *, ep, bins=18):
    curves = []
    for c in sorted([c for c in cells if c["e_over_p"] == ep], key=lambda c: c["vpp"]):
        t = np.asarray(c["transit_us_all"])
        if t.size < 10:
            continue
        u = t / np.median(t)
        h, edges = np.histogram(u, bins=bins, range=(0.8, 1.3), density=True)
        mid = 0.5 * (edges[1:] + edges[:-1])
        curves.append(ProfileCurve(f"Vpp {c['vpp']:g} (n {t.size}, skew "
                                   f"{c['transit_skew']:+.2f})", mid, h,
                                   {"drawstyle": "steps-mid"}))
    return curves


def build_all(cells, outdir, *, ref_ep=1.51, atd_ep=0.35):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    op = op_transit(cells[0])
    cur, _ = transit_ratio_curves(cells, ref_ep=ref_ep)
    f1 = render_profiles(cur, xlabel="RF amplitude Vpp (V)",
                         ylabel="transit / ideal transit  (t_ref x E_ref/E)",
                         title="G1: transit time vs RF amplitude",
                         subtitle=op, hmarks=[(1.0, "ideal (Eq. 2)")])
    f1.savefig(outdir / "g1_transit_ratio.png", dpi=DEFAULT_DPI)
    f2 = render_profiles(path_curves(cells), xlabel="RF amplitude Vpp (V)",
                         ylabel="integrated path per axial mm (mm/mm)",
                         title="G1: total path length flown vs RF amplitude",
                         subtitle=op)
    f2.savefig(outdir / "g1_path_per_mm.png", dpi=DEFAULT_DPI)
    f3 = render_profiles(atd_curves(cells, ep=atd_ep),
                         xlabel="transit / median transit",
                         ylabel="density",
                         title=f"G1: arrival-time shape at E/P {atd_ep:g} V/cm/Torr",
                         subtitle=op)
    f3.savefig(outdir / "g1_atd_shape.png", dpi=DEFAULT_DPI)
    return [outdir / n for n in ("g1_transit_ratio.png", "g1_path_per_mm.png",
                                 "g1_atd_shape.png")]


In [ ]:
# ==== INLINED from g1b_analysis.py: residence-time referee
# analysis, verbatim minus __main__.
"""G1b -- residence time per ring period: is the RF delay a dwell (tail)
phenomenon, an ion-to-ion heterogeneity ("multiple paths"), or a uniform
slow-down?  Consumes g1_run --residence JSON.

Referee: 1-D drift-diffusion first-passage over one pitch L is inverse
Gaussian, IG(mu = L/v_d, lam = L^2 / (2 D)), with D = kT v_d / (q E)
(Einstein) -- both v_d and E taken from the SAME cell, so the referee is
what a smooth-field drift tube would give at this cell's measured drift
speed.  Excess density at tau >> mu, over the IG, is dwell.

Decomposition per cell:
  within-ion variance of tau (period to period), between-ion variance of
  the per-ion mean tau, and the iid-renewal expectation of the latter
  (within / n_periods).  Ratio > 1 = persistent slow ions.
  Pearson r(tau_k, r_period_k): dips near the wall predict tau up with r.
  Renewal check: Var(transit) vs n_periods x Var(tau).
"""
from __future__ import annotations
import json, math, sys
from pathlib import Path
import numpy as np

from ion_gym.viz.design_maps import ProfileCurve, render_profiles

KB_EV = 8.617333e-5


def ig_pdf(t, mu, lam):
    t = np.asarray(t, float)
    out = np.zeros_like(t)
    m = t > 0
    out[m] = np.sqrt(lam / (2 * np.pi * t[m] ** 3)) * np.exp(
        -lam * (t[m] - mu) ** 2 / (2 * mu ** 2 * t[m]))
    return out


def analyse_cell(c, *, T_k=298.0, charge=2, L_mm=1.5):
    taus = [np.asarray(t) for t in c["tau_us_per_ion"] if len(t) > 0]
    rps = [np.asarray(r) for r in c["r_period_per_ion"] if len(r) > 0]
    n_per = min(len(t) for t in taus)
    T = np.array([t[:n_per] for t in taus])          # ions x periods
    R = np.array([r[:n_per] for r in rps])
    tau_all, r_all = T.ravel(), R.ravel()
    mu = tau_all.mean()
    E_v_mm = c["E_ideal_v_mm"]
    v_d = L_mm / mu                                   # mm/us at THIS cell
    D = KB_EV * T_k * v_d / (charge * E_v_mm)         # mm^2/us
    lam = L_mm ** 2 / (2 * D)
    within = T.var(axis=1, ddof=1).mean()
    between = T.mean(axis=1).var(ddof=1)
    per_ion_mean = T.mean(axis=1)
    rho = float(np.corrcoef(tau_all, r_all)[0, 1])
    transit = T.sum(axis=1)
    thr = mu + 4 * math.sqrt(mu ** 3 / lam)           # IG sigma = sqrt(mu^3/lam)
    return dict(
        vpp=c["vpp"], e_over_p=c["e_over_p"], n_ions=T.shape[0],
        n_periods=n_per, tau_mean_us=float(mu), tau_sd_us=float(tau_all.std()),
        ig_mu=float(mu), ig_lam=float(lam), ig_sd=float(math.sqrt(mu ** 3 / lam)),
        skew_tau=float(((tau_all - mu) ** 3).mean() / tau_all.std() ** 3),
        skew_ig=float(3 * math.sqrt(mu / lam)),
        frac_beyond_4sig=float((tau_all > thr).mean()),
        frac_beyond_4sig_ig=float(1 - _ig_cdf(thr, mu, lam)),
        within_var=float(within), between_var=float(between),
        between_over_iid=float(between / (within / n_per)),
        rho_tau_r=rho,
        var_transit=float(transit.var(ddof=1)),
        var_transit_renewal=float(n_per * within),
        tau_all=tau_all, r_all=r_all, per_ion_mean=per_ion_mean)


def _ig_cdf(x, mu, lam):
    from math import erf, sqrt, exp
    Phi = lambda z: 0.5 * (1 + erf(z / sqrt(2)))
    a = sqrt(lam / x)
    return Phi(a * (x / mu - 1)) + exp(2 * lam / mu) * Phi(-a * (x / mu + 1))


def op_residence(c):
    n_per = c["plane_b_ring"] - c["plane_a_ring"]
    return (f"Allen&Bush cell i.d. {c['bore_id_mm']:g} mm, {c['n_rings']}-ring model, "
            f"E/P {c['e_over_p']:g} V/cm/Torr, {c['rf_freq_hz']/1e6:g} MHz, "
            f"{c['p_torr']:g} Torr {c['gas']} {c['T_k']:g} K, GRGDS2+ "
            f"{c['ion_mass_da']:g} Da z=+{c['ion_charge']}, HS1 dt {c['dt_ns']:g} ns, "
            f"pitch {c['L_mm']/n_per:g} mm, {c['n_transit']} ions x {n_per} periods; "
            f"referee = inverse-Gaussian first passage at each cell's own v_d (Einstein D)")


def build_figs(cells, outdir, *, ep):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    cs = sorted([c for c in cells if c["e_over_p"] == ep], key=lambda c: c["vpp"])
    an = [analyse_cell(c, T_k=c["T_k"], charge=c["ion_charge"],
                       L_mm=c["L_mm"] / (c["plane_b_ring"] - c["plane_a_ring"]))
          for c in cs]
    # SURVIVAL curves, not histograms (the density
    # view was hard to read). S(u) = fraction of pitch crossings that
    # take LONGER than u times the cell's mean -- "one crossing in a
    # hundred is slower than 1.4x the mean" reads directly off the
    # curve, and a trapping tail would lift the measured curve above
    # the referee at large u. No binning, so no binning artifacts.
    curves = []
    u_max = 2.0
    for c, a in zip(cs, an):
        u = np.sort(a["tau_all"] / a["tau_mean_us"])
        surv = 1.0 - np.arange(1, u.size + 1) / u.size
        u_max = max(u_max, float(u[-1]) * 1.05)
        curves.append(ProfileCurve(f"Vpp {c['vpp']:g}", u,
                                   np.maximum(surv, 0.5 / u.size),
                                   {"drawstyle": "steps-post"}))
    # IG referee for the lowest-Vpp cell (shape ~ same across cells)
    a0 = an[0]
    u_ref = np.linspace(0.5, u_max, 400)
    s_ref = np.array([1.0 - _ig_cdf(uu * a0["ig_mu"], a0["ig_mu"],
                                    a0["ig_lam"]) for uu in u_ref])
    curves.append(ProfileCurve("drift-diffusion referee (IG)",
                               u_ref, np.maximum(s_ref, 1e-5),
                               {"color": "k", "ls": "--"}))
    f1 = render_profiles(curves,
                         xlabel="residence time per pitch, in units of the mean",
                         ylabel="fraction of crossings slower than this",
                         logy=True,
                         title=f"G1b: how often is a pitch crossing slow? E/P {ep:g}",
                         subtitle=op_residence(cs[0]))
    f1.savefig(outdir / f"g1b_tau_dist_ep{str(ep).replace('.', 'p')}.png", dpi=DEFAULT_DPI)
    # tau vs radius, binned
    curves = []
    rb = np.linspace(0.0, 0.5 * cs[0]["bore_id_mm"], 11)
    for c, a in zip(cs, an):
        idx = np.digitize(a["r_all"], rb) - 1
        m = np.array([a["tau_all"][idx == k].mean() if (idx == k).sum() > 20 else np.nan
                      for k in range(len(rb) - 1)])
        curves.append(ProfileCurve(f"Vpp {c['vpp']:g} (rho {a['rho_tau_r']:+.2f})",
                                   0.5 * (rb[1:] + rb[:-1]), m / a["tau_mean_us"],
                                   {"marker": "o"}))
    f2 = render_profiles(curves, xlabel="mean radius during the period (mm)",
                         ylabel="residence time / cell mean",
                         title=f"G1b: residence time vs radial position, E/P {ep:g}",
                         subtitle=op_residence(cs[0]),
                         vmarks=[(0.5 * cs[0]["bore_id_mm"], f"ring bore (i.d. {cs[0]['bore_id_mm']:g} mm)")])
    f2.savefig(outdir / f"g1b_tau_vs_r_ep{str(ep).replace('.', 'p')}.png", dpi=DEFAULT_DPI)
    return an


def table(an):
    print("E/P   Vpp  n   <tau> us  sd/IGsd  skew/IG      >4sig %/IG   betw/iid  rho(tau,r)  Var(T)/renewal")
    for a in an:
        print(f"{a['e_over_p']:5} {a['vpp']:4g} {a['n_ions']:3d}  {a['tau_mean_us']:7.1f}  "
              f"{a['tau_sd_us']/a['ig_sd']:.3f}   {a['skew_tau']:+.2f}/{a['skew_ig']:+.2f}   "
              f"{a['frac_beyond_4sig']*100:.2f}/{a['frac_beyond_4sig_ig']*100:.2f}     "
              f"{a['between_over_iid']:.2f}     {a['rho_tau_r']:+.3f}     "
              f"{a['var_transit']/a['var_transit_renewal']:.2f}")


## 1. The device and the process — see it before the statistics

The geometry the solver actually received (electrode mask, not a redrawing), with three example GRGDS2+ trajectories at 200 Vpp, E/P 0.507. Note the transverse RF wiggle (micromotion) — enormous path, and, as §2 shows, time-neutral.

In [ ]:
from ion_gym.physics.sim_build import build_run
from ion_gym.viz import viz_core as V
from ion_gym.progress import quote, track
# rendered live on every run (a solve plus a few example ions, seconds):
# a banked PNG could show a deck the parameter cell no longer describes
spec_geo = rf_drift_cell_spec(**DEVICE, rf_vpp=max(VPP), e_over_p=SHOW_E_OVER_P,
                              n_ions=N_EXAMPLE, dt_ns=DT_NS, seed=SEED)
t_build = time.time()
model_geo, fly_geo, cols_geo, births_geo = build_run(spec_geo, record_budget_gb=RECORD_BUDGET_GB)
quote("live geometry panel: example flights after a measured build",
      N_EXAMPLE, per_item_s=(time.time() - t_build))
trajs = []
fates = []
for i in track(range(N_EXAMPLE), label="example flights"):
    tr, summ = fly_geo(i)
    trajs.append(tr)
    fates.append(summ['kind'])
sc = V.scene_from_simspec(
    spec_geo, model_geo, field='phi', trajs=trajs, fates=fates,
    title=(f'SRIG drift cell: {N_RINGS} rings, {BORE_ID_MM:g} mm i.d., '
           f'E/P {SHOW_E_OVER_P}, {max(VPP)} Vpp @ {RF_FREQ_HZ/1e6:g} MHz, '
           f'{P_TORR:g} Torr {GAS}, GRGDS2+ {ION_MASS_DA:g} Da z=+{ION_Z}, '
           f'dt {DT_NS} ns -- {N_EXAMPLE} example trajectories'))
png_geo = OUT / '24_geometry_trajectories.png'
V.render_mpl(sc).savefig(png_geo, dpi=DEFAULT_DPI)
display(Image(str(png_geo)))

## 1.5 The mechanism, drawn in the fields

Before any statistics, here is the physics the campaigns test, taken from the solved fields the ions fly in (the same arrays, not an illustration). Three views, each per RF amplitude:

**The axial corrugation.** The effective potential on the axis, with the linear DC ramp subtracted so the ripple is visible: discrete rings corrugate the pseudopotential *on the axis itself* with the ring pitch, and the well depth grows as $V_{pp}^2$. These are the ripples a drifting ion crosses once per pitch. Their depth against $kT$ is printed.

**The radial wall.** The same effective potential across the bore at a gap plane: the confinement wall that keeps ions off the rings, again growing as $V_{pp}^2$. This is why the RF-off cells lose most ions to the walls while every 100–200 $V_{pp}$ ion survives — and why the surviving cloud's mean radius shrinks with amplitude.

**The DC drive itself.** The axial DC field at three radii: discrete rings also corrugate the *drive*, weakly on axis and strongly near the rings, while its pitch-average equals the ideal ladder field at every radius. Section 1.6 asks how much delay each of these ripples should produce, and at which radius.

In [ ]:
# the three field views, from the solved model at each Vpp; the
# geometry is identical across amplitudes, so the base solve is done
# once and reused (basis superposition), making this cell cheap
from ion_gym.viz.design_maps import ProfileCurve, render_profiles

Z_WIN_MM = (30.0, 36.0)  # four mid-stack pitches, away from the ends
R_CUTS_MM = (0.0, 1.25, 2.0)

from ion_gym.progress import quote, track

def model_at(v):
    spec_v = rf_drift_cell_spec(**DEVICE, rf_vpp=v, e_over_p=SHOW_E_OVER_P,
                                n_ions=1, dt_ns=DT_NS, seed=SEED)
    model_v, fly_v, cols_v, births_v = build_run(spec_v, record_budget_gb=RECORD_BUDGET_GB)
    return model_v

t_build = time.time()
models = {VPP[0]: model_at(VPP[0])}
quote("field models per Vpp (first build measured; the base solve is "
      "shared, so the rest reuse it)", len(VPP) - 1,
      per_item_s=(time.time() - t_build))
for v in track(VPP[1:], label="Vpp models"):
    models[v] = model_at(v)

op_f = (f"Allen&Bush cell i.d. {BORE_ID_MM:g} mm, {N_RINGS} rings, pitch "
        f"{PITCH_MM} mm, {RF_FREQ_HZ/1e6:g} MHz, E/P {SHOW_E_OVER_P:g} "
        f"V/cm/Torr ladder, GRGDS2+ {ION_MASS_DA:g} Da z=+{ION_Z}; effective potential = "
        f"DC + Dehmelt pseudopotential from the solved field")

# -- axial corrugation: U_eff(z, r=0) minus its linear ramp ------------
curves, depths = [], {}
for v in VPP:
    z, r_full, pe, _em = models[v].pe_surface(mz=ION_MASS_DA, charge=ION_Z)
    i_axis = int(np.argmin(np.abs(r_full)))
    sel = (z >= Z_WIN_MM[0]) & (z <= Z_WIN_MM[1])
    cut = pe[sel, i_axis]
    slope, intercept = np.polyfit(z[sel], cut, 1)
    ripple = cut - (slope * z[sel] + intercept)
    depths[v] = float(ripple.max() - ripple.min())
    curves.append(ProfileCurve(f"{v:g} Vpp", z[sel], ripple * 1e3, {}))
print("on-axis corrugation depth (peak to trough):")
for v in VPP:
    print(f"  {v:4g} Vpp: {depths[v]*1e3:7.2f} meV = "
          f"{depths[v]/KT_EV:6.2f} kT")
f = render_profiles(curves, xlabel="z (mm)",
                    ylabel="on-axis effective potential, ramp removed (meV)",
                    title="the axial corrugation the DC must drag ions over",
                    subtitle=op_f,
                    hmarks=[(KT_EV * 1e3, "kT (298 K)")])
f.savefig(OUT / '24_mech_axial_corrugation.png', dpi=DEFAULT_DPI)
display(Image(str(OUT / '24_mech_axial_corrugation.png')))

# -- radial wall: U_eff(r) at a gap plane, axis value removed ----------
curves = []
for v in VPP:
    z, r_full, pe, _em = models[v].pe_surface(mz=ION_MASS_DA, charge=ION_Z)
    i_axis = int(np.argmin(np.abs(r_full)))
    z_gap = Z_WIN_MM[0] + PITCH_MM / 2.0   # mid-gap between two rings
    iz = int(np.argmin(np.abs(z - z_gap)))
    sel_r = (r_full >= 0.0) & (r_full <= BORE_ID_MM / 2.0)
    wall = pe[iz, sel_r] - pe[iz, i_axis]
    curves.append(ProfileCurve(f"{v:g} Vpp", r_full[sel_r], wall, {}))
f = render_profiles(curves, xlabel="r (mm)",
                    ylabel="effective potential above the axis (eV)",
                    title="the radial wall at a gap plane",
                    subtitle=op_f, logy=True,
                    hmarks=[(KT_EV, "kT (298 K)")],
                    vmarks=[(BORE_ID_MM / 2.0, "ring bore")])
f.savefig(OUT / '24_mech_radial_wall.png', dpi=DEFAULT_DPI)
display(Image(str(OUT / '24_mech_radial_wall.png')))

# -- the DC drive vs radius (Vpp-independent: DC basis only) -----------
z, r_full, phi, _em = models[VPP[0]].potential_image(rf_phase=None)
ez = -np.gradient(phi, z, axis=0)          # V/mm; z is in mm
sel = (z >= Z_WIN_MM[0]) & (z <= Z_WIN_MM[0] + 2 * PITCH_MM)
curves = []
for rc in R_CUTS_MM:
    ir = int(np.argmin(np.abs(r_full - rc)))
    curves.append(ProfileCurve(f"r = {r_full[ir]:.2f} mm", z[sel],
                               ez[sel, ir], {}))
e_ideal = SHOW_E_OVER_P * P_TORR / 10.0
pitch_mean = float(ez[sel, int(np.argmin(np.abs(r_full)))].mean())
print(f"pitch-averaged on-axis DC field {pitch_mean:.4f} V/mm vs ideal "
      f"ladder {e_ideal:.4f} V/mm")
f = render_profiles(curves, xlabel="z (mm)",
                    ylabel="axial DC field (V/mm)",
                    title="the DC drive is corrugated too, but its "
                          "pitch average is the ideal ladder at every r",
                    subtitle=op_f,
                    hmarks=[(e_ideal, "ideal ladder field")])
f.savefig(OUT / '24_mech_dc_field.png', dpi=DEFAULT_DPI)
display(Image(str(OUT / '24_mech_dc_field.png')))

## 1.6 Predicted slowdown at each radius

The ripples in section 1.5 were drawn on the axis. The question now is how much each ripple should delay an ion, and how that depends on where in the cross section the ion travels.

Start with an ion held at one radius r. Along the axis it feels the effective potential U(z, r): the DC ramp plus the pseudopotential, the latter reduced slightly for gas damping (the factor (ωτ)²/(1+(ωτ)²) with τ = mK/q, printed below). The ion's momentum relaxes in well under a microsecond, while one pitch takes tens of microseconds, so its axial motion is drift plus diffusion in U. For a ramp with a periodic ripple on top, the mean velocity of that motion has an exact closed form (Stratonovich, for a tilted periodic potential). It needs only U over one pitch, the ramp force F and kT. Mobility and cross section drop out of the ratio to free drift. The cell below folds the solved U(z, r) over the 33 measured pitches, removes the ramp, and evaluates that formula at every lattice radius. The result is the fractional slowdown s(r) = t(r)/t_free − 1.

Two things to look for in the figure. Near the axis the ripple is shallow and s(r) is small. Away from the axis the ripple deepens quickly, because the RF field grows toward the rings, and once the ripple's steepest slope exceeds the ramp's, the summed potential gains real minima and s(r) climbs steeply. Ions there can be held. The table marks the smallest radius where that happens for each cell.

An ion does not stay at one radius, so s(r) alone does not predict a transit. Two averages bracket the answer. If an ion keeps roughly the same radius while it crosses a pitch, the expected delay is s(r) averaged over the radii at which the flown ions actually crossed pitches, which section 2b computes from the flights. If the ion instead samples the whole cross section in thermal equilibrium within each pitch, its axial motion sees one free-energy profile A(z) = −kT ln ∫ exp(−U(z, r)/kT) r dr (the Fick–Jacobs reduction), and the same formula gives one delay per cell with no flights at all. In this cell radial diffusion across half a millimetre takes about as long as one pitch crossing, so neither limit is guaranteed, and the flights decide.

In [ ]:
# predictor machinery: pure functions of the solved potential.
# Nothing here flies an ion; the fields come from the same solves the
# flights use (section 1.5 models), re-weighted per E/P and Vpp.
from scipy.interpolate import CubicSpline
from ion_gym.physics.ccs import low_field_mobility_m2Vs
from ion_gym.physics.constants import E_CHG, KG_AMU


def fold_pitch_profile(z_mm, u_ev, z_a_mm, z_b_mm, pitch_mm):
    '''Fold a 1-D potential over whole pitches between two planes.

    Returns the pitch-phase grid (mm), the folded potential with the
    linear ramp removed (eV, absolute offset kept so radii compare),
    the ramp force F = -dU/dz over the window (eV/mm), and the largest
    pitch-to-pitch deviation in SHAPE from the folded mean (eV), each
    pitch compared after removing its own mean. (A residual linear trend
    of a few tenths of a meV across the 33 pitches shifts whole pitches
    up or down without changing their shape, so it is excluded here.)'''
    h = float(z_mm[1] - z_mm[0])
    ia = int(round((z_a_mm - z_mm[0]) / h))
    ib = int(round((z_b_mm - z_mm[0]) / h))
    n_per = int(round(pitch_mm / h))
    if abs(n_per * h - pitch_mm) > 1e-9 or (ib - ia) % n_per:
        raise ValueError(
            f"window {z_a_mm}->{z_b_mm} mm is not a whole number of "
            f"{pitch_mm} mm pitches on the {h} mm lattice")
    span = z_mm[ib] - z_mm[ia]
    F = (u_ev[ia] - u_ev[ib]) / span
    seg = u_ev[ia:ib] + F * (z_mm[ia:ib] - z_mm[ia])
    stack = seg.reshape(-1, n_per)
    folded = stack.mean(axis=0)
    shape = stack - stack.mean(axis=1, keepdims=True)
    spread = float(np.abs(shape - (folded - folded.mean())).max())
    phase = np.arange(n_per) * h
    return phase, folded, float(F), spread


def fine_profile(phase_mm, p_ev, pitch_mm, n_fine):
    '''Periodic cubic spline of a folded profile on n_fine points, with
    its derivative (eV/mm).'''
    spl = CubicSpline(np.append(phase_mm, pitch_mm), np.append(p_ev, p_ev[0]),
                      bc_type="periodic")
    x = np.arange(n_fine) * (pitch_mm / n_fine)
    return x, spl(x), spl(x, 1)


def washboard_slowdown(p_fine_ev, pitch_mm, F_ev_mm, kT_ev):
    '''t / t_free - 1 for overdamped drift + diffusion in p(x) - F x with
    p periodic over one pitch (Stratonovich mean velocity). The ramp
    integral is exact per fine cell and the inner integral is a circular
    convolution done by FFT. A flat p returns 0 to rounding.'''
    n = p_fine_ev.size
    d = pitch_mm / n
    b = 1.0 / kT_ev
    p = p_fine_ev - p_fine_ev.min()
    c = b * F_ev_mm
    kern = np.exp(-c * np.arange(n) * d) * (-np.expm1(-c * d)) / c
    inner = np.real(np.fft.ifft(np.fft.fft(np.exp(-b * p)) * np.fft.fft(kern)))
    denom = d * np.sum(np.exp(b * p) * inner)
    free = pitch_mm * (-np.expm1(-c * pitch_mm)) / c
    return float(denom / free - 1.0)


def gas_factor(device, rf_freq_hz):
    '''(w tau)^2 / (1 + (w tau)^2) with tau = m K / q from the Mason-Schamp
    mobility of the same cross section the hard-sphere kernel flies.'''
    spec0 = rf_drift_cell_spec(**device, rf_vpp=0.0, e_over_p=1.0, n_ions=1)
    # the ion's MASS in Da (review C-11: the function takes mass for every
    # charge; passing m/z gave a lighter reduced mass)
    K = low_field_mobility_m2Vs(spec0.collisions, device["ion_mass_da"],
                                device["ion_charge"])
    tau = device["ion_mass_da"] * KG_AMU * K / (device["ion_charge"] * E_CHG)
    wt = 2 * np.pi * rf_freq_hz * tau
    return wt ** 2 / (1 + wt ** 2), tau


print("predictor control, flat potential:",
      washboard_slowdown(np.zeros(N_FINE), PITCH_MM, 0.35, KT_EV))

In [ ]:
# s(r) for every (E/P, Vpp) cell, the first radius with real minima,
# and the Fick-Jacobs cross-section average. The DC basis is the same at
# every Vpp, so U = U_DC + gamma * (U_total - U_DC) re-weights two solves.
GAMMA, TAU_S = gas_factor(DEVICE, RF_FREQ_HZ)
GAMMA_USED = GAMMA if GAS_DAMPED_PSEUDO else 1.0
print(f"momentum relaxation time tau = {TAU_S*1e6:.3f} us; gas factor "
      f"{GAMMA:.4f} ({'applied' if GAS_DAMPED_PSEUDO else 'NOT applied'})")


def model_for(e_over_p, vpp):
    spec_m = rf_drift_cell_spec(**DEVICE, rf_vpp=vpp, e_over_p=e_over_p,
                                n_ions=1, dt_ns=DT_NS, seed=SEED)
    model_m, fly_m, cols_m, births_m = build_run(spec_m, record_budget_gb=RECORD_BUDGET_GB)
    return spec_m, model_m


def predict_cell(e_over_p, vpp):
    spec_dc, model_dc = model_for(e_over_p, 0.0)
    spec_rf, model_rf = model_for(e_over_p, vpp)
    z, r_full, pe_dc, em = model_dc.pe_surface(mz=ION_MASS_DA, charge=ION_Z)
    z2, r_full2, pe_rf, em2 = model_rf.pe_surface(mz=ION_MASS_DA, charge=ION_Z)
    u = pe_dc + GAMMA_USED * (pe_rf - pe_dc)
    half = 0.5 * spec_rf.geometry.electrodes[0].shapes[0].params["width_mm"]
    z_a = spec_rf.geometry.electrodes[PLANE_A_RING - 1].shapes[0].params["x_mm"] + half
    z_b = spec_rf.geometry.electrodes[PLANE_B_RING - 1].shapes[0].params["x_mm"] + half
    keep = (r_full >= -1e-12) & (r_full <= R_GRID_MAX_MM + 1e-9)
    radii = r_full[keep]
    cols_r = np.nonzero(keep)[0]
    s = np.zeros(radii.size)
    pfine = np.zeros((radii.size, N_FINE))
    well_r = None
    spread_max = 0.0
    depth_axis = None
    F_axis = None
    for k, j in enumerate(cols_r):
        phase, p, F, spread = fold_pitch_profile(z, u[:, j], z_a, z_b, PITCH_MM)
        x_f, p_f, dp_f = fine_profile(phase, p, PITCH_MM, N_FINE)
        pfine[k] = p_f
        s[k] = washboard_slowdown(p_f, PITCH_MM, F, KT_EV)
        if radii[k] <= R_PLOT_MAX_MM:
            spread_max = max(spread_max, spread)
        if well_r is None and dp_f.max() > F:
            well_r = float(radii[k])
        if k == 0:
            depth_axis = float(p.max() - p.min())
            F_axis = F
    # Fick-Jacobs: free energy over the cross section, weight r dr
    wall_kT = (pfine[-1].mean() - pfine[0].mean()) / KT_EV
    if wall_kT < FJ_WALL_MIN_KT:
        s_fj = None
        fj_note = (f"refused: radial wall at {radii[-1]:.2f} mm is only "
                   f"{wall_kT:.1f} kT, so the average depends on where it is cut")
    else:
        w = np.where(radii > 0, radii, 0.25 * (radii[1] - radii[0]))
        b = 1.0 / KT_EV
        lw = np.log(w)[:, None] - b * (pfine - pfine.min())
        top = lw.max(axis=0)
        a_fj = -KT_EV * (top + np.log(np.exp(lw - top).sum(axis=0)))
        s_fj = washboard_slowdown(a_fj, PITCH_MM, F_axis, KT_EV)
        fj_note = f"radial wall {wall_kT:.0f} kT"
    return dict(e_over_p=e_over_p, vpp=vpp, r=radii, s=s, well_r=well_r,
                s_fj=s_fj, fj_note=fj_note, depth_axis_ev=depth_axis,
                F_ev_mm=F_axis, spread_ev=spread_max)


PRED = {}
cells_pred = [(ep, v) for ep in E_OVER_P for v in VPP]
t_first = time.time()
PRED[cells_pred[0]] = predict_cell(*cells_pred[0])
quote("field predictions per cell (first cell measured, and the solves "
      "are shared, so the rest re-weight)", len(cells_pred) - 1,
      per_item_s=(time.time() - t_first))
for key in track(cells_pred[1:], label="prediction cells"):
    PRED[key] = predict_cell(*key)

op_pred = (f"Allen&Bush cell i.d. {BORE_ID_MM:g} mm, {N_RINGS}-ring model, "
           f"pitch {PITCH_MM} mm, {RF_FREQ_HZ/1e6:g} MHz, {P_TORR:g} Torr "
           f"{GAS} {T_GAS_K:g} K, GRGDS2+ {ION_MASS_DA:g} Da z=+{ION_Z}; "
           f"U = DC + {GAMMA_USED:.3f} x Dehmelt pseudopotential from the "
           f"solved field; overdamped drift-diffusion at fixed radius")
print("E/P    Vpp   axis ripple      s(0)     s(0.5mm)  s(1.0mm)  minima from   Fick-Jacobs")
for (ep, v), pr in PRED.items():
    s_at = [np.interp(rr, pr["r"], pr["s"]) for rr in (0.0, 0.5, 1.0)]
    well = f"{pr['well_r']:.2f} mm" if pr["well_r"] is not None else "none"
    fj = f"{pr['s_fj']*100:+.2f}%" if pr["s_fj"] is not None else pr["fj_note"]
    print(f"{ep:5}  {v:4g}  {pr['depth_axis_ev']*1e3:6.2f} meV "
          f"({pr['depth_axis_ev']/KT_EV:4.2f} kT)  {s_at[0]*100:+7.3f}%  "
          f"{s_at[1]*100:+7.3f}%  {s_at[2]*100:+7.3f}%  {well:>10}   {fj}")
print(f"largest pitch-to-pitch deviation in ripple shape within "
      f"{R_PLOT_MAX_MM} mm of the axis: "
      f"{max(pr['spread_ev'] for pr in PRED.values())*1e3:.3f} meV "
      f"(kT = {KT_EV*1e3:.1f} meV)")

for ep in E_OVER_P:
    curves = []
    for v in VPP:
        if v == 0:
            continue
        pr = PRED[(ep, v)]
        sel = (pr["r"] <= R_PLOT_MAX_MM) & (pr["s"] * 100 > 1e-4)
        well = (f", minima from {pr['well_r']:.2f} mm"
                if pr["well_r"] is not None and pr["well_r"] <= R_PLOT_MAX_MM
                else "")
        if not sel.any():
            print(f"E/P {ep:g}, {v:g} Vpp: predicted slowdown below 1e-4 % at "
                  f"every radius up to {R_PLOT_MAX_MM} mm, not drawn")
            continue
        curves.append(ProfileCurve(f"{v:g} Vpp{well}", pr["r"][sel],
                                   pr["s"][sel] * 100, {"marker": "."}))
    f = render_profiles(curves, xlabel="radius held fixed (mm)",
                        ylabel="predicted slowdown t/t_free - 1 (%)",
                        title=f"predicted slowdown at each radius, E/P {ep:g} "
                              f"V/cm/Torr (values below 1e-4 % not drawn)",
                        subtitle=op_pred, logy=True)
    png = OUT / f"24_pred_slowdown_vs_r_ep{str(ep).replace('.', 'p')}.png"
    f.savefig(png, dpi=DEFAULT_DPI)
    display(Image(str(png)))

## 2.0 How many ions each manuscript figure needs

Each figure answers a question at a stated precision, and that precision sets the ion count. The cell below derives the counts from the targets in the parameter cell instead of choosing them by hand.

* **Delay against Vpp, flown and predicted (sections 2 and 2b).** A cell's delay is its mean transit divided by the RF-off reference cell's, scaled by E/P. Its standard error combines both cells' relative errors, each the transit's coefficient of variation divided by √n. The spread of transit times comes from diffusion over the 49.5 mm measured length, σ_t/t = √(2kT/(zeEL)) (Einstein), and the measured spread replaces that estimate wherever a banked cell exists. Giving the cell and the reference equal shares of the target error requires n = 2(σ_t/t)²/SE².
* **Residence-time tails (section 3).** The figure asks whether high RF amplitude raises the fraction of slow pitch crossings above the drift-diffusion referee. Resolving the planning excess at TAIL_SIGMA standard errors requires N = TAIL_SIGMA²·p/(Δp)² crossings, and each ion contributes 33.
* **Arrival curves and radial density (section 4)** reuse the same flights. **RF-off cells below the reference field** only report survival, because almost no ion crosses the cell, so they get N_TRANSMISSION births.

Each cell takes the largest requirement that applies to it. The requirements count ions that cross both planes, so the births are that number divided by the cell's transit fraction: measured where a banked campaign holds the cell, otherwise the planning values in the parameter cell (RF off loses most ions to the rings).

In [ ]:
# derive the ion count per cell; nothing flies here
LENGTH_MM = (PLANE_B_RING - PLANE_A_RING) * PITCH_MM


def einstein_cv(e_over_p):
    '''sigma_t / t for drift-diffusion over LENGTH_MM at this E/P.'''
    e_field = e_over_p * P_TORR / 10.0            # V/mm
    return float(np.sqrt(2.0 * KT_EV / (ION_Z * e_field * LENGTH_MM)))


def round_up(n, step=50):
    return int(step * math.ceil(n / step))


# measured spread and survival, where any banked campaign (this one or
# an earlier one) holds the cell with enough transits to estimate them
measured = {}
for path in sorted(DATA.glob("g1*.json")):
    for c in json.loads(path.read_text()):
        if c.get("n_transit", 0) >= 20 and "transit_sd_us" in c:
            measured[(c["e_over_p"], c["vpp"])] = dict(
                cv=c["transit_sd_us"] / c["transit_mean_us"],
                survival=c["n_transit"] / c["n_ions"], source=path.name)


def cv_for(ep, v):
    if (ep, v) in measured:
        return measured[(ep, v)]["cv"], measured[(ep, v)]["source"]
    return einstein_cv(ep), "Einstein estimate"


def survival_for(ep, v):
    if (ep, v) in measured:
        return measured[(ep, v)]["survival"]
    return SURVIVAL_RF_OFF_PLAN if v == 0 else SURVIVAL_RF_ON_PLAN


cv_ref, src_ref = cv_for(REF_E_OVER_P, REF_VPP)
n_tail = round_up(TAIL_SIGMA ** 2 * TAIL_P_FLOWN
                  / (TAIL_P_FLOWN - TAIL_P_IG) ** 2 / (PLANE_B_RING - PLANE_A_RING))
N_IONS = {}
print(f"target delay SE {TARGET_DELAY_SE*100:.2f}% | tail excess at "
      f"{TAIL_SIGMA:g} SE needs {n_tail} ions | measured length {LENGTH_MM} mm")
print("E/P    Vpp   sigma_t/t  (source)                 survival  n_delay  n_tail  births  role")
for ep in E_OVER_P:
    for v in VPP:
        cv, src = cv_for(ep, v)
        if v == 0 and ep != REF_E_OVER_P:
            n_delay = 0
            n_t = 0
            n_used = N_TRANSMISSION
            role = "survival only"
        else:
            # the targets count TRANSITS; births = transits / survival
            n_delay = round_up(2 * cv ** 2 / TARGET_DELAY_SE ** 2)
            n_t = n_tail if v > 0 else 0
            n_used = round_up(max(n_delay, n_t) / survival_for(ep, v))
            role = "reference" if (ep, v) == (REF_E_OVER_P, REF_VPP) else "delay"
        N_IONS[(ep, v)] = n_used
        print(f"{ep:5}  {v:4g}  {cv:.4f}  ({src:22s})  {survival_for(ep, v):8.2f}  "
              f"{n_delay:7d}  {n_t:6d}  {n_used:6d}  {role}")
n_total = sum(N_IONS.values())
print(f"TOTAL: {n_total} ions over {len(N_IONS)} cells")

In [ ]:
# QUOTE ONLY -- nothing of the campaign runs here. One ion per E/P is
# timed at the highest Vpp (a confined ion flies the full length, the
# most expensive case) on THIS machine, after an untimed warm-up flight.
# The projection assumes the per-ion time scales ideally over the
# workers; the run cell's progress lines report the measured rate.
from ion_gym.progress import quote, track


def time_one_ion(e_over_p, vpp):
    spec_q = rf_drift_cell_spec(**DEVICE, rf_vpp=vpp, e_over_p=e_over_p,
                                n_ions=2, dt_ns=DT_NS, seed=SEED)
    spec_q.integration.rec_every = max(
        int(spec_q.integration.t_max_us / (DT_NS * 1e-3) / ROWS_PER_ION), 1)
    model_q, fly_q, cols_q, births_q = build_run(spec_q, record_budget_gb=RECORD_BUDGET_GB)
    fly_q(0)
    t0 = time.perf_counter()
    fly_q(1)
    return time.perf_counter() - t0


per_ion_s = {}
t_probe = time.time()
per_ion_s[E_OVER_P[0]] = time_one_ion(E_OVER_P[0], max(VPP))
# the remaining probes fly slower ions (lower E/P), so this quote is a floor
quote("timing probes at the remaining E/P (build + two flights each)",
      len(E_OVER_P) - 1, per_item_s=(time.time() - t_probe))
for ep in track(E_OVER_P[1:], label="timing probes"):
    per_ion_s[ep] = time_one_ion(ep, max(VPP))
for ep in E_OVER_P:
    print(f"E/P {ep:g}: {per_ion_s[ep]:.1f} s per ion on one worker", flush=True)
todo = {}
if CAMPAIGN_FILE.exists():
    banked = {(c["e_over_p"], c["vpp"]) for c in json.loads(CAMPAIGN_FILE.read_text())}
else:
    banked = set()
for key, n in N_IONS.items():
    if key not in banked:
        todo[key] = n
cpu_s = sum(n * per_ion_s[key[0]] for key, n in todo.items())
print(f"QUOTE: {len(todo)} cells still to fly ({sum(todo.values())} ions), "
      f"{cpu_s/3600:.1f} worker-hours -> about {cpu_s/N_WORKERS/3600:.1f} h "
      f"on {N_WORKERS} workers (RF-off cells priced as if confined: an upper bound)")
print("RUN =", RUN, "-- set RUN=True in the parameter cell to spend it")

## 2. Campaign G1 — transit against Vpp, and what moves with it

One cell = one (E/P, Vpp) pair with the ion count from section 2.0. The runner records, per ion, the **exact kernel time integrals** of path length, axial field and kinetic energy (per-step accumulators: a stroboscopic `rec_every` average aliases against the RF and was measured to bias ⟨E_ax⟩ by +11%, which is why these exist), plus the time and mean radius of every ring-pitch crossing between the two planes. The table's columns are the candidate explanations side by side. The campaign file is rewritten after each cell, so an interrupted run resumes where it stopped.

In [ ]:
RUN = True

In [ ]:
# RUN CELL -- spends the quote above when RUN is True. Cells already
# banked with the same settings are skipped; a banked cell with different
# settings is refused rather than mixed.
if RUN:
    run_campaign(CAMPAIGN_FILE, device=DEVICE, n_ions_by_cell=N_IONS,
                 dt_ns=DT_NS, e_over_p=E_OVER_P, vpp=VPP,
                 plane_a_ring=PLANE_A_RING, plane_b_ring=PLANE_B_RING,
                 seed=SEED, rows_per_ion=ROWS_PER_ION, n_workers=N_WORKERS)
if not CAMPAIGN_FILE.exists():
    raise RuntimeError(
        f"no campaign file {CAMPAIGN_FILE.name} under {DATA}: set RUN=True "
        f"in the parameter cell to fly it (the quote cell prices it). Earlier "
        f"campaign files (g1_pass2, g1b) are not read: they were flown at a "
        f"different step and with the generator's default ion settings.")
all_cells = json.loads(CAMPAIGN_FILE.read_text())
for c in all_cells:
    wanted = dict(DEVICE, dt_ns=DT_NS, seed=SEED, plane_a_ring=PLANE_A_RING,
                  plane_b_ring=PLANE_B_RING,
                  n_ions=N_IONS.get((c["e_over_p"], c["vpp"]), c["n_ions"]))
    diff = cell_mismatch(c, wanted)
    if diff:
        raise RuntimeError(f"{CAMPAIGN_FILE.name}: cell E/P {c['e_over_p']:g}, "
                           f"{c['vpp']:g} Vpp differs from the parameter cell: {diff}")
cells = [c for c in all_cells if c.get("n_transit", 0) > 0]
have = {(c["e_over_p"], c["vpp"]) for c in all_cells}
missing = sorted({(ep, v) for ep in E_OVER_P for v in VPP} - have)
if missing:
    print(f"NOT YET FLOWN ({len(missing)} cells): {missing} -- the analysis "
          f"below covers the {len(have)} banked cells only")
for c in all_cells:
    if c.get("n_transit", 0) == 0:
        print(f"E/P {c['e_over_p']:g}, {c['vpp']:g} Vpp: no ion crossed both "
              f"planes ({c['n_ions']} births, fates {c['fates']}); survival only")
paths = build_all(cells, OUT, ref_ep=REF_E_OVER_P, atd_ep=min(E_OVER_P))
ref_cell = [c for c in cells if c["e_over_p"] == REF_E_OVER_P and c["vpp"] == REF_VPP]
if len(ref_cell) != 1:
    raise RuntimeError(f"the reference cell E/P {REF_E_OVER_P}, {REF_VPP} Vpp "
                       f"is needed for every delay and is not banked with transits")
ref_cell = ref_cell[0]
ref_rel_se = ref_cell["transit_se_us"] / ref_cell["transit_mean_us"]
print("E/P    Vpp   n      delay (t/t_ideal - 1)   path/axial-mm  <Eax>/E_ideal  <r> mm")
for c in sorted(cells, key=lambda c: (-c["e_over_p"], c["vpp"])):
    if c["n_transit"] < 5:
        continue
    t_ideal = ref_cell["transit_mean_us"] * REF_E_OVER_P / c["e_over_p"]
    ratio = c["transit_mean_us"] / t_ideal
    se = ratio * math.hypot(c["transit_se_us"] / c["transit_mean_us"], ref_rel_se)
    print(f"{c['e_over_p']:5}  {c['vpp']:4g}  {c['n_transit']:5d}  "
          f"{(ratio-1)*100:+7.3f} +- {se*100:.3f} %   {c['path_per_axial_mm']:6.2f}"
          f"         {c['e_ax_over_ideal']:.4f}       {c['r_mean_mm']:.3f}")
for p in paths:
    display(Image(str(p)))

**Reading this table.** Two of the candidate explanations make different predictions here. If extra path length cost time, the delay would follow path per axial mm, which rises with Vpp at every E/P alike. If the delay comes from the drift velocity, it should grow as E/P falls, because the same ripple matters more against a weaker ramp. The ⟨E_ax⟩/E_ideal column tracks the delay almost by construction (drift velocity is mobility times the time-averaged field), so agreement between those two columns confirms the bookkeeping but does not identify a cause. Section 2b tests the cause against the field prediction. Heating appears as the kinetic-energy proxy in the per-cell printout above the table.

## 2b. Flown delay against the field prediction

Section 1.6 predicted a slowdown at every radius from the solved fields alone. Here that prediction is turned into one number per flown cell in three ways and set against the flown delay.

* **Axis only:** s(0), the delay an ion would have if it never left the axis.
* **Crossing-weighted radius:** s(r) averaged over every pitch crossing of the flown ions, each at its own mean radius during that pitch. This is the slow-radial-exchange limit. The crossing radius is averaged over the pitch, which smooths micromotion, so it reads slightly low near the axis.
* **Fick–Jacobs:** the thermal-equilibrium cross-section average from section 1.6 (the fast-radial-exchange limit), where the radial wall is deep enough for it to be defined.

All three are referenced to the RF-off reference cell exactly as the flown delay is, so a common bias cancels the same way in both. How to read the outcome: flown points on the axis-only curve would mean the centre line carries the effect. Points on one of the cross-section averages would mean the effective potential over the occupied cross section accounts for the slowdown, and the two averages show which radial-exchange limit applies. Points above every curve would mean something outside the pseudopotential picture adds delay, and micromotion coupled to collisions would be the first candidate to test. The printout also gives the fraction of pitch crossings made at radii where the potential has real minima, which says directly whether flown ions visit regions where they could be held.

In [ ]:
# flown vs predicted delay per cell; no flights here


def crossing_radii(c):
    return np.concatenate([np.asarray(rr, float) for rr in c["r_period_per_ion"]
                           if len(rr)])


# An unconfined (RF-off) cell lets a few ions diffuse nearly to the
# rings, past the predictor grid. The fixed-radius drift model means
# nothing against the metal (its minima there are many eV deep, and an
# ion in them splats rather than drifts), so such crossings are EXCLUDED
# from the weighted mean as a reported skip, never extrapolated. The
# exclusion is harmless exactly where it happens: only unconfined cells
# reach those radii, and their prediction is flat near zero. The bound
# below refuses if the excluded share ever grows large enough to matter.
MAX_BEYOND_GRID_FRACTION = 0.01


def crossing_weighted(c, pr):
    '''Mean predicted slowdown over the flown pitch crossings, and the
    full crossing-radius array (before any exclusion).'''
    r = crossing_radii(c)
    beyond = r > pr["r"].max()
    frac = float(beyond.mean())
    if frac > MAX_BEYOND_GRID_FRACTION:
        raise ValueError(
            f"E/P {c['e_over_p']:g}, {c['vpp']:g} Vpp: {int(beyond.sum())} "
            f"of {r.size} pitch crossings ({frac:.1%}) lie beyond the "
            f"predictor grid at {pr['r'].max():.2f} mm (farthest "
            f"{r.max():.2f} mm). Too many to exclude without moving the "
            f"average: this cell's occupancy needs understanding, not a "
            f"wider grid.")
    if beyond.any():
        print(f"E/P {c['e_over_p']:g}, {c['vpp']:g} Vpp: excluded "
              f"{int(beyond.sum())} of {r.size} pitch crossings "
              f"({frac:.3%}) beyond the predictor grid at "
              f"{pr['r'].max():.2f} mm (farthest {r.max():.2f} mm) -- "
              f"see the exclusion note above this function")
    kept = r[~beyond]
    return float(np.mean(np.interp(kept, pr["r"], pr["s"]))), r


x_ref, r_ref = crossing_weighted(ref_cell, PRED[(REF_E_OVER_P, REF_VPP)])
print(f"reference cell's own predicted slowdown (crossing-weighted): "
      f"{x_ref*100:+.4f}%  (every prediction below is divided by 1 + this)")
COMPARISON = []
print("E/P    Vpp   flown              axis only   crossing-weighted   Fick-Jacobs   crossings in minima region")
for c in sorted(cells, key=lambda c: (-c["e_over_p"], c["vpp"])):
    if c["n_transit"] < 5:
        continue
    key = (c["e_over_p"], c["vpp"])
    pr = PRED[key]
    t_ideal = ref_cell["transit_mean_us"] * REF_E_OVER_P / c["e_over_p"]
    ratio = c["transit_mean_us"] / t_ideal
    se = ratio * math.hypot(c["transit_se_us"] / c["transit_mean_us"], ref_rel_se)
    s_cw, radii_c = crossing_weighted(c, pr)
    axis = (1 + pr["s"][0]) / (1 + x_ref) - 1
    cw = (1 + s_cw) / (1 + x_ref) - 1
    fj = None if pr["s_fj"] is None else (1 + pr["s_fj"]) / (1 + x_ref) - 1
    in_minima = (float((radii_c >= pr["well_r"]).mean())
                 if pr["well_r"] is not None else 0.0)
    COMPARISON.append(dict(e_over_p=key[0], vpp=key[1], flown=ratio - 1,
                           flown_se=se, axis=axis, crossing=cw, fj=fj,
                           in_minima=in_minima, n=c["n_transit"]))
    fj_txt = f"{fj*100:+7.3f}%" if fj is not None else "   n/a  "
    print(f"{key[0]:5}  {key[1]:4g}  {(ratio-1)*100:+7.3f}+-{se*100:.3f}%   "
          f"{axis*100:+7.3f}%    {cw*100:+7.3f}%           {fj_txt}      "
          f"{in_minima*100:.3f}%")

op_cmp = op_transit(ref_cell) + (f"; predictions from the solved fields, "
                                 f"pseudopotential x {GAMMA_USED:.3f}")
for ep in E_OVER_P:
    rows_ep = sorted([row for row in COMPARISON if row["e_over_p"] == ep],
                     key=lambda row: row["vpp"])
    if not rows_ep:
        print(f"E/P {ep:g}: no flown cells yet, figure skipped")
        continue
    vx = [row["vpp"] for row in rows_ep]
    curves = [ProfileCurve("flown (+- 1 SE)", vx,
                           [row["flown"] * 100 for row in rows_ep],
                           {"marker": "o", "ls": "none", "color": "k"},
                           yerr=[row["flown_se"] * 100 for row in rows_ep]),
              ProfileCurve("predicted, axis only", vx,
                           [row["axis"] * 100 for row in rows_ep],
                           {"ls": ":", "marker": "."}),
              ProfileCurve("predicted, crossing-weighted radius", vx,
                           [row["crossing"] * 100 for row in rows_ep],
                           {"ls": "-", "marker": "."})]
    fj_rows = [row for row in rows_ep if row["fj"] is not None]
    if fj_rows:
        curves.append(ProfileCurve("predicted, Fick-Jacobs equilibrium",
                                   [row["vpp"] for row in fj_rows],
                                   [row["fj"] * 100 for row in fj_rows],
                                   {"ls": "--", "marker": "."}))
    f = render_profiles(curves, xlabel="RF amplitude Vpp (V)",
                        ylabel="delay over ideal drift, t/t_ideal - 1 (%)",
                        title=f"flown delay against the field prediction, "
                              f"E/P {ep:g} V/cm/Torr",
                        subtitle=op_cmp, hmarks=[(0.0, "ideal drift")])
    png = OUT / f"24_pred_vs_flown_ep{str(ep).replace('.', 'p')}.png"
    f.savefig(png, dpi=DEFAULT_DPI)
    display(Image(str(png)))

## 3. Residence-time referee

Sharper question: *how does an ion cross one ring pitch?* Per-pitch first-passage times are compared against an **inverse-Gaussian referee** computed from each cell's own drift speed and Einstein diffusion, which is the exact prediction for smooth drift-diffusion with no free parameters. If RF created traps or a slow-ion class, it would show here. These residence times come from the same flights as section 2.

**What this assesses, and why this referee.** Section 2 measures *that* RF amplitude lengthens transit. This section asks whether the *shape* of the slowing is what pure drift-diffusion predicts, ion by ion. The assessed quantity is the per-ion **residence time τ** in each ring period: the first-passage time of a particle advected at v_d and diffusing with D across a slab one period wide. First passage of drifted Brownian motion has a closed form, the **inverse-Gaussian distribution** IG(μ = L/v_d, λ = L²/2D). That is why it is the referee: it is not a fit family chosen for convenience but the analytic answer for the null hypothesis "nothing retains ions but drift and diffusion."

**How to read a deviation.** If the measured τ distribution sits on the IG curve, tail included, the cell's slowdown is a uniform reduction of drift speed with no hidden mechanism. A **heavy right tail above IG** means retention beyond drift-diffusion: ions dwelling in real minima of the potential, which section 1.6 locates away from the axis. A **narrower-than-IG** distribution would mean the tracer under-disperses (a physics bug, not chemistry). The survival curves below are where each verdict is read: each curve answers "what fraction of pitch crossings take longer than u times the mean?", so a trapping tail would lift a measured curve above the dashed referee at large u, and an under-dispersing tracer would fall below it everywhere.

In [ ]:
# residence analysis on the section 2 flights: every cell recorded its
# pitch crossings, so nothing new flies here
bcells = [c for c in cells if c.get("tau_us_per_ion")]
if not bcells:
    raise RuntimeError(
        f"no cell in {CAMPAIGN_FILE.name} carries residence records; the "
        f"section 2 runner writes them for every cell, so this file was not "
        f"written by it")
allan = []
for ep in sorted({c["e_over_p"] for c in bcells}, reverse=True):
    allan += build_figs(bcells, OUT, ep=ep)
table(allan)
# display what THIS run built, not a hardcoded name list
for p in sorted(OUT.glob("g1b_tau_dist_ep*.png")) + \
         sorted(OUT.glob("g1b_tau_vs_r_ep*.png")):
    display(Image(str(p)))

### The experimental picture: later arrivals, tighter cloud

Two distributions carry the whole result in the units an instrument reports. **First figure:** cumulative arrival curves at the lowest E/P, one per RF amplitude — "by time $t$, what fraction of the ions has arrived?" The curves displace to later times as $V_{pp}$ rises, and the shift of the median is read where each curve crosses 0.5. A cumulative curve is used instead of a binned distribution because a histogram's bin noise can bury a shift of a few percent, while the cumulative curve needs no bins. (The RF-off cell is excluded with its count printed: without confinement almost no ion survives this E/P.) **Second figure:** where those same ions spent their transit radially — the number density per unit cross-section area, over every recorded ring-pitch crossing, binned in equal-area annuli. It peaks on the axis at every amplitude (a plain histogram in $r$ would peak off-axis purely because an annulus at $r$ has area $2\pi r\,dr$ — an area factor, not a hollow cloud) and pulls sharply onto the axis as the $V_{pp}^2$ wall of §1.5 deepens. One honest caveat on the innermost bin: the variable is each crossing's *mean* radius over 1.5 mm of travel, and an ion that crosses the axis still has a nonzero mean radius, so the very first bin sits slightly low relative to the instantaneous density. Read together with §1.5: a tighter cloud rides closer to the axis where the pseudopotential corrugation is, and arrives later for it.

In [ ]:
# both panels come from the banked campaigns -- no new flights.
# Arrival times: the G1 cells' per-ion transits at the lowest E/P.
# Radii: the per-pitch mean radii (micromotion averaged), one value
# per ion per ring period, from the same flights.
ep_show = E_OVER_P[-1]
atd_cells = sorted([c for c in cells if c['e_over_p'] == ep_show],
                   key=lambda c: c['vpp'])
# cumulative arrival curves, not density histograms: with 50-100 ions
# per cell a binned density is choppy and buries a 4% shift, while the
# cumulative curve is binning-free and the shift reads as a horizontal
# displacement. "By time t, what fraction has arrived?"
t15 = None
curves = []
op_cell = None
for c in atd_cells:
    t = np.sort(np.asarray(c['transit_us_all'])) / 1e3
    if t.size < 5:
        print(f"{c['vpp']:g} Vpp at E/P {ep_show:g}: only {t.size} of "
              f"{c['n_ions']} ions survive -- too few for a "
              f"distribution, excluded from the figure")
        continue
    if t15 is None:
        t15 = t.mean()
    if op_cell is None:
        op_cell = c
    frac = np.arange(1, t.size + 1) / t.size
    curves.append(ProfileCurve(
        f"{c['vpp']:g} Vpp -- mean {t.mean():.3f} ms "
        f"({(t.mean() / t15 - 1) * 100:+.1f}% vs {op_cell['vpp']:g} Vpp, "
        f"n {t.size})",
        t, frac, {"drawstyle": "steps-post"}))
# this panel's data is the G1 transit campaign, so its operating point
# is stated directly (the shared _op helper describes the G1b residence
# runs and their referee, which is not what is plotted here)
op_atd = op_transit(op_cell)
f = render_profiles(curves, xlabel="arrival time (ms)",
                    ylabel="fraction of ions arrived",
                    title=f"arrival curves shift later as RF amplitude "
                          f"rises (E/P {ep_show:g})",
                    subtitle=op_atd,
                    hmarks=[(0.5, "median")])
f.savefig(OUT / '24_atd_vs_vpp.png', dpi=DEFAULT_DPI)
display(Image(str(OUT / '24_atd_vs_vpp.png')))

# radial CROSS-SECTION DENSITY per Vpp, from the G1b per-pitch mean
# radii. A histogram in r peaks off-axis even for an axis-centered
# cloud, because an annulus at radius r has area 2 pi r dr -- the
# area factor, not hollowness. Dividing by the annulus area gives the
# density per unit cross-section area, which peaks on the axis as
# intuition expects; bins are equal in r^2 (equal area), so every bin
# carries comparable statistics and the division adds no noise.
occ = sorted([a for a in allan if a['e_over_p'] == ep_show],
             key=lambda a: a['vpp'])
edges_r2 = np.linspace(0.0, (BORE_ID_MM / 2.0) ** 2, 26)
area_mm2 = np.pi * np.diff(edges_r2)
r_edges = np.sqrt(edges_r2)
curves = []
for a in occ:
    counts, _ = np.histogram(np.asarray(a['r_all']) ** 2, bins=edges_r2)
    dens = counts / counts.sum() / area_mm2      # fraction per mm^2
    # draw each bin as its true [r_lo, r_hi] step, so the curve starts
    # at the axis instead of the first bin's midpoint
    x_step = np.repeat(r_edges, 2)[1:-1]
    y_step = np.repeat(dens, 2)
    curves.append(ProfileCurve(
        f"{a['vpp']:g} Vpp -- mean r {np.mean(a['r_all']):.2f} mm "
        f"({a['n_ions']} ions x {a['n_periods']} pitches)",
        x_step, y_step, {}))
f = render_profiles(curves,
                    xlabel="radius (mm)",
                    ylabel="fraction of pitch crossings per mm^2",
                    title=f"cross-section density: the cloud is centered "
                          f"on the axis and tightens as Vpp^2 "
                          f"(E/P {ep_show:g})",
                    subtitle=op_residence(op_cell),
                    vmarks=[(BORE_ID_MM / 2.0, "ring bore")])
f.savefig(OUT / '24_radial_occupancy.png', dpi=DEFAULT_DPI)
display(Image(str(OUT / '24_radial_occupancy.png')))

**Reading the residence table.** At RF off and at low amplitude the measured spread, skew and tail should sit on the referee: that certifies the tracer's transport statistics before anything is concluded at high amplitude. At high amplitude, compare three columns. The spread ratio (sd over the referee's sd) and the 4σ tail fraction say whether slow crossings exceed drift-diffusion. The between-ion over iid ratio says whether some ions are persistently slow (a value near 1 means every ion is slowed alike). The correlation of τ with radius says whether slow crossings happen off axis, where section 1.6 places the real minima.

## 4. Verdict, and what each outcome would mean

The verdict is read from three places, in this order.

1. **Section 2b.** Whether the flown delays at each E/P fall on the axis-only prediction, on a cross-section average, or above every prediction. The first would say the centre line carries the effect. The second would say the effective potential over the occupied cross section accounts for it, with the closer of the two averages naming the radial-exchange regime. The third would say a mechanism outside the pseudopotential picture adds delay.
2. **The minima column of section 2b and the residence table.** If almost no crossings occur at radii with real minima and the residence tails sit near the referee, the slowdown is uneven drift over a ripple, not trapping, and the word "wells" overstates it for this device. A measurable fraction of crossings in the minima region together with a heavy residence tail would instead mean some ions are held, which is the trapping picture.
3. **Path length and heating (section 2 table).** Path per axial mm should rise with Vpp at every E/P while the delay does not follow it, and the kinetic-energy proxy should stay near thermal. Either one tracking the delay would reopen that explanation.

Open items stated rather than assumed: the cross section is the RF-cell measurement (141 Å², Allen et al. 2016) rather than the drift-tube value (145.6 Å², Kemper 2009), a 3% choice that cancels to first order in every ratio here; the pseudopotential is used only for the prediction and never for flight; and the prediction folds the 33 interior pitches into one profile (the largest pitch-to-pitch shape deviation is printed in section 1.6).